In [ ]:
# -*- coding: utf-8 -*-
"""
============================================================
Intra-Dataset KNN Distribution Analysis
============================================================

Core Metrics:
-------------
1. **k-NN Radius**
   - Definition: Distance from each point to its k-th nearest neighbor
   - Physical meaning: Reflects local data density around the point
   - Smaller radius → Higher local density (data is dense)
   - Larger radius → Lower local density (data is sparse)

2. **Coverage Rate**
   - Definition: Proportion of points in dataset B covered by k-NN balls of dataset A
   - Calculation: For each point b in B, if there exists point a in A such that dist(a,b) <= radius_A[a], then b is covered
   - Coverage(A→B) = 1 - hole_rate(A_to_B)

3. **Hole Rate**
   - Definition: Proportion of uncovered points
   - hole_rate = holes / total_points

Data Storage Format (logs/wicompass/knn_coverage/{dataset}/k{k}_{dedup|nodedup}.{json|h5}):
---------------------------------------------------------------------------------
JSON file contains:
  - A_to_B / B_to_A: Coverage statistics (radius_mean, radius_p95, radius_max, holes, hole_rate)
  - metadata: Dataset info, k value, deduplication info, etc.

H5 file contains:
  - coverage_indices/: A_covered_by_B, A_uncovered_by_B, B_covered_by_A, B_uncovered_by_A
  - intra_knn/: A_distances (N_A, k), A_indices (N_A, k), B_distances (N_B, k), B_indices (N_B, k)
  - knn_mappings/: A_to_B_nearest, B_to_A_nearest
  - radii/: A_knn_radii (distance to k-th neighbor), B_knn_radii
  - processed/raw/: Raw token data

Note: When k > 16, intra_knn data is not stored (to save space)
"""

import os
import json
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ========= Data Path Configuration =========
KNN_COVERAGE_DIR = Path("../../logs/wicompass/knn_coverage")

# ========= Color Configuration =========
PALETTE = {
    2: '#1e90ff',   # Blue
    4: '#ffbb00',   # Orange-yellow
    6: '#ff5080',   # Pink
    8: '#a7426d',   # Purple-red
    10: '#ff3c10',  # Orange-red
    12: '#3cb371',  # Soft green
    16: '#282828',  # Dark gray
    32: '#9467bd',  # Purple
    64: '#8c564b',  # Brown
    128: '#e377c2', # Pink
}

# ========= Global Style =========
plt.rcdefaults()
plt.rcParams.update({
    "figure.figsize": [5, 3],
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "grid.linestyle": "--",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 2.5,
    "font.size": 16,
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "legend.fontsize": "medium",
    "legend.frameon": True,
})

# ========= Data Loading Functions =========
def load_knn_json(dataset: str, k: int, dedup: bool = True) -> dict:
    """
    Load k-NN coverage statistics from JSON file
    
    Args:
        dataset: Dataset name ('mmfi', 'mmbody', 'wicompass')
        k: k value
        dedup: Whether to use deduplicated version
    
    Returns:
        Dictionary containing statistics
    """
    suffix = "dedup" if dedup else "nodedup"
    json_path = KNN_COVERAGE_DIR / dataset / f"k{k}_{suffix}.json"
    
    if not json_path.exists():
        raise FileNotFoundError(f"File not found: {json_path}")
    
    with open(json_path, 'r') as f:
        return json.load(f)


def load_knn_radii(dataset: str, k: int, side: str, dedup: bool = True) -> np.ndarray:
    """
    Load k-NN radius data from H5 file
    
    Args:
        dataset: Dataset name ('mmfi', 'mmbody', 'wicompass')
        k: k value
        side: 'A' (AMASS) or 'B' (target dataset)
        dedup: Whether to use deduplicated version
    
    Returns:
        k-NN radius array (N,)
    """
    suffix = "dedup" if dedup else "nodedup"
    h5_path = KNN_COVERAGE_DIR / dataset / f"k{k}_{suffix}.h5"
    
    if not h5_path.exists():
        raise FileNotFoundError(f"File not found: {h5_path}")
    
    with h5py.File(h5_path, 'r') as f:
        radii = f[f"radii/{side}_knn_radii"][:]
    
    return radii


def load_intra_knn_distances(dataset: str, k: int, side: str, dedup: bool = True) -> np.ndarray:
    """
    Load complete intra-dataset k-NN distance matrix from H5 file
    
    Args:
        dataset: Dataset name
        k: k value (must be <= 16, otherwise data is not stored)
        side: 'A' or 'B'
        dedup: Whether to use deduplicated version
    
    Returns:
        Distance matrix (N, k)
    """
    if k > 16:
        raise ValueError(f"k={k} > 16: intra_knn data not stored to save space")
    
    suffix = "dedup" if dedup else "nodedup"
    h5_path = KNN_COVERAGE_DIR / dataset / f"k{k}_{suffix}.h5"
    
    if not h5_path.exists():
        raise FileNotFoundError(f"File not found: {h5_path}")
    
    with h5py.File(h5_path, 'r') as f:
        if "intra_knn" not in f:
            raise KeyError(f"File does not contain intra_knn data: {h5_path}")
        distances = f[f"intra_knn/{side}_distances"][:]
    
    return distances


def get_available_k_values(dataset: str, dedup: bool = True) -> list:
    """Get all computed k values for a dataset"""
    suffix = "dedup" if dedup else "nodedup"
    data_dir = KNN_COVERAGE_DIR / dataset
    
    if not data_dir.exists():
        return []
    
    k_values = []
    for f in data_dir.glob(f"k*_{suffix}.json"):
        try:
            k = int(f.stem.split('_')[0][1:])  # Extract 8 from "k8_dedup"
            k_values.append(k)
        except ValueError:
            continue
    
    return sorted(k_values)


# ========= CDF Computation Function =========
def cdf_by_histogram(x: np.ndarray, bins: int = 1024) -> tuple:
    """Approximate CDF using histogram accumulation (equal-width bins)"""
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return x, x
    counts, edges = np.histogram(x, bins=bins)
    cdf = np.cumsum(counts) / counts.sum()
    return edges[1:], cdf


# ========= Test Loading =========
print("Available datasets and k values:")
for ds in ["mmfi", "mmbody", "wicompass"]:
    k_vals = get_available_k_values(ds)
    if k_vals:
        print(f"  {ds}: k = {k_vals}")

In [ ]:
# -*- coding: utf-8 -*-
"""
Coverage Statistics Summary Table
- Display coverage rate changes across different k values
- Compare A→B and B→A coverage
"""

import pandas as pd

def summarize_coverage(dataset: str, k_values: list = None, dedup: bool = True) -> pd.DataFrame:
    """
    Summarize coverage statistics for a given dataset
    
    Args:
        dataset: Dataset name
        k_values: List of k values, None for all available values
        dedup: Whether to use deduplicated version
    
    Returns:
        DataFrame containing statistics
    """
    available_k = get_available_k_values(dataset, dedup)
    if k_values is None:
        k_values = available_k
    else:
        k_values = [k for k in k_values if k in available_k]
    
    records = []
    for k in k_values:
        try:
            data = load_knn_json(dataset, k, dedup)
            meta = data.get("metadata", {})
            
            record = {
                "k": k,
                "A_size": meta.get("unique_counts", {}).get("A", "N/A"),
                "B_size": meta.get("unique_counts", {}).get("B", "N/A"),
                "A→B_coverage": f"{(1 - data['A_to_B']['hole_rate']) * 100:.2f}%",
                "B→A_coverage": f"{(1 - data['B_to_A']['hole_rate']) * 100:.2f}%",
                "A_radius_mean": f"{data['A_to_B']['radius_mean']:.4f}",
                "B_radius_mean": f"{data['B_to_A']['radius_mean']:.4f}",
            }
            records.append(record)
        except Exception as e:
            print(f"⚠️ k={k} load failed: {e}")
    
    return pd.DataFrame(records)


# ========= MMFi Coverage Summary =========
print("=" * 60)
print("MMFi Dataset Coverage Summary")
print("Note: A=AMASS, B=MMFi")
print("A→B_coverage: Proportion of MMFi covered by AMASS")
print("B→A_coverage: Proportion of AMASS covered by MMFi")
print("=" * 60)
df_mmfi = summarize_coverage("mmfi", k_values=[2, 4, 8, 16, 32, 64, 128])
display(df_mmfi)

# ========= mmBody Coverage Summary =========
print("\n" + "=" * 60)
print("mmBody Dataset Coverage Summary")
print("Note: A=AMASS, B=mmBody")
print("=" * 60)
df_mmbody = summarize_coverage("mmbody", k_values=[2, 4, 6, 8, 10, 12])
display(df_mmbody)

# NRI (Normalized Redundancy Index)

## Why Normalization?
Raw k-NN radius $r_k$ is not comparable across different k values: larger k naturally leads to larger radius.

## NRI Definition
Based on k-NN density estimation theory: $f_i \propto \frac{k}{(n-1) \cdot r_k^{d}}$

**NRI** = $\frac{(n-1) \cdot r_k^{d_{\text{eff}}}}{k}$

- Smaller NRI → Higher local density → More redundant data
- Larger NRI → Lower local density → More unique/sparse data

## Effective Dimension $d_{\text{eff}}$ Estimation
Using the relationship $\log r_k = \frac{1}{d} \log k + C$, solve via linear regression on median radii across multiple k values.

In [ ]:
# -*- coding: utf-8 -*-
"""
NRI (Normalized Redundancy Index) Analysis
- Normalize k-NN radii to comparable scale via effective dimension estimation
- Smaller NRI = higher local density (more redundant), larger NRI = sparser (more unique)
"""

# ========= Effective Dimension Estimation =========
def estimate_d_eff(radii_by_k: dict) -> float:
    """
    Estimate effective dimension d_eff from k-NN radii across multiple k values
    
    Theory: log(r_k) = (1/d) * log(k) + C
    Solve via linear regression: slope s = 1/d, so d_eff = 1/s
    """
    if len(radii_by_k) < 2:
        return 8.0  # Default value
    
    ks = np.array(sorted(radii_by_k.keys()), dtype=np.float64)
    medians = np.array([np.median(radii_by_k[int(k)]) for k in ks], dtype=np.float64)
    
    # Log transform
    x = np.log(ks + 1e-12)
    y = np.log(medians + 1e-12)
    
    # Robust regression: remove endpoints
    order = np.argsort(x)
    x, y = x[order], y[order]
    if x.size >= 4:
        x, y = x[1:-1], y[1:-1]
    
    # Linear regression: y = s*x + b, where s = 1/d
    s, b = np.polyfit(x, y, deg=1)
    
    if s <= 0:
        return 8.0
    
    d_eff = 1.0 / s
    return float(np.clip(d_eff, 1.0, 128.0))


def compute_nri(radii: np.ndarray, k: int, n: int, d_eff: float) -> np.ndarray:
    """
    Compute NRI (Normalized Redundancy Index)
    NRI_i = ((n-1) * r_k^d_eff) / k
    """
    r = np.maximum(radii, 1e-12)
    return ((n - 1) * (r ** d_eff)) / float(k)


def load_multi_k_radii(dataset: str, side: str, k_values: list, dedup: bool = True) -> tuple:
    """Load radii data for multiple k values"""
    side = side.upper()
    radii_by_k = {}
    n = None
    
    for k in k_values:
        try:
            radii = load_knn_radii(dataset, k, side, dedup)
            radii = radii[np.isfinite(radii) & (radii > 0)]
            if radii.size > 0:
                radii_by_k[k] = radii
                if n is None:
                    n = radii.size
                else:
                    n = min(n, radii.size)
        except Exception as e:
            print(f"⚠️ {dataset} k={k} load failed: {e}")
    
    # Align lengths
    for k in radii_by_k:
        if radii_by_k[k].size > n:
            radii_by_k[k] = radii_by_k[k][:n]
    
    return radii_by_k, n or 0


def plot_nri_cdf(dataset: str, side: str, k_values: list, 
                 dedup: bool = True, xlim: tuple = None, 
                 title: str = None, save_path: str = None,
                 d_eff_override: float = None):
    """
    Plot NRI CDF distribution
    
    Args:
        d_eff_override: If provided, use this d_eff instead of estimating.
                        Useful for WiCompass which should use AMASS's d_eff.
    
    X-axis determination:
    - Computed from 0th and 98th percentiles of NRI values across all k values
    - min(xlim) = min of all 0th percentiles
    - max(xlim) = max of all 98th percentiles
    - This ensures all curves are visible while excluding extreme outliers
    """
    side = side.upper()
    
    # Get available k values
    available_k = get_available_k_values(dataset, dedup)
    k_values = [k for k in k_values if k in available_k]
    
    if not k_values:
        print(f"⚠️ {dataset} has no available k value data")
        return None, None
    
    print(f"📊 Plotting {dataset} ({side}) NRI CDF, k = {k_values}")
    
    # Load data
    radii_by_k, n = load_multi_k_radii(dataset, side, k_values, dedup)
    
    if not radii_by_k or n <= 0:
        print(f"⚠️ {dataset} has no available data")
        return None, None
    
    # Determine d_eff: use override if provided, otherwise estimate
    if d_eff_override is not None:
        d_eff = d_eff_override
        print(f"📐 Using override d_eff = {d_eff:.2f}")
    else:
        d_eff = estimate_d_eff(radii_by_k)
        print(f"📐 Estimated d_eff = {d_eff:.2f}")
    
    fig, ax = plt.subplots()
    
    xlim_vals = []
    for k in sorted(radii_by_k.keys()):
        radii = radii_by_k[k]
        nri = compute_nri(radii, k, n, d_eff)
        x, y = cdf_by_histogram(nri)
        
        color = PALETTE.get(k, '#333333')
        ax.plot(x, y, label=f"k={k}", color=color, linestyle="-")
        
        # Statistics
        q25, q50, q75 = np.percentile(nri, [25, 50, 75])
        print(f"  k={k}: median={q50:.4g}, Q1={q25:.4g}, Q3={q75:.4g}")
        
        # Collect xlim values: 0th and 98th percentiles
        xlim_vals.extend(np.percentile(nri, [0, 98]))
    
    ax.set_ylabel("CDF")
    ax.set_ylim(0, 1.05)
    
    if xlim:
        ax.set_xlim(xlim)
    else:
        # X-axis range: union of [0th, 98th] percentiles across all k values
        ax.set_xlim(min(xlim_vals), max(xlim_vals))
    
    # if title:
    #     ax.set_title(title)
    
    ax.legend(loc="lower right", framealpha=0.85)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"✅ Saved: {save_path}")
    
    plt.show()
    
    return d_eff, n


# ========= Configuration =========
K_VALUES = [2, 4, 6, 8, 10, 12]
DEDUP = False  # Use non-deduplicated version (nodedup)

# ========= AMASS NRI Analysis (using A side from mmbody) =========
# AMASS uses its own estimated d_eff, and we save it for WiCompass
print("=" * 60)
print("AMASS Dataset NRI Analysis")
print("=" * 60)
d_eff_amass, _ = plot_nri_cdf("mmbody", side="A", k_values=K_VALUES, dedup=DEDUP,
                               title="AMASS", save_path="nri_cdf_AMASS.pdf")

# ========= MMFi NRI Analysis =========
# MMFi uses its own estimated d_eff
print("\n" + "=" * 60)
print("MMFi Dataset NRI Analysis")
print("=" * 60)
plot_nri_cdf("mmfi", side="B", k_values=K_VALUES, dedup=DEDUP,
             title="MMFi", save_path="nri_cdf_MMFi.pdf", d_eff_override=d_eff_amass)

# ========= mmBody NRI Analysis =========
# mmBody uses its own estimated d_eff
print("\n" + "=" * 60)
print("mmBody Dataset NRI Analysis")
print("=" * 60)
plot_nri_cdf("mmbody", side="B", k_values=K_VALUES, dedup=DEDUP,
             title="mmBody", save_path="nri_cdf_mmBody.pdf", d_eff_override=d_eff_amass)

# ========= WiCompass NRI Analysis =========
# WiCompass is sampled from AMASS, so it uses AMASS's d_eff
print("\n" + "=" * 60)
print("WiCompass Dataset NRI Analysis")
print(f"(Using AMASS d_eff = {d_eff_amass:.2f} because WiCompass is sampled from AMASS)")
print("=" * 60)
plot_nri_cdf("wicompass", side="B", k_values=K_VALUES, dedup=DEDUP,
             title="WiCompass", save_path="nri_cdf_WiCompass.pdf",
             d_eff_override=d_eff_amass)